In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path
import tbparse

EXPERIMENT_NAME = "ablation_oplora_scaled_damping"
EXPERIMENT_DIR = Path("..") / "output" / EXPERIMENT_NAME
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"


In [ ]:
def get_experiment_df(dedup="last"):
    dfs_list = []
    for file in EXPERIMENT_DIR.glob("*"):
        run_dir = Path(file)
        if not run_dir.is_dir():
            continue  # skip non-directories
        if run_dir.name.startswith("_"):
            continue  # skip hidden directories
        print(run_dir)

        # df = tbparse.SummaryReader(run_dir, pivot=True).scalars
        df = tbparse.SummaryReader(run_dir, pivot=False).scalars
        if dedup == "first":
            df = df.drop_duplicates(subset=["step", "tag"], keep="first")
        elif dedup == "last":
            df = df.drop_duplicates(subset=["step", "tag"], keep="last")
        elif dedup == "mean":
            df = df.groupby(["step", "tag"], as_index=False)["value"].mean()
        elif dedup == "max":
            df = df.groupby(["step", "tag"], as_index=False)["value"].max()
        else:
            raise ValueError(f"Unknown dedup method: {dedup}")

        df = df.pivot(index="step", columns="tag", values="value").reset_index()

        df["Optimizer"] = "SGD" if "sgd" in str(file.name) else "AdamW"
        df["Method"] = "Scaled OPLoRA"
        keyvals = file.name.split("_")
        df["Seed"] = int(keyvals[0].replace("seed=", ""))
        df["LR"] = float(keyvals[1].replace("lr=", ""))
        df["K-FAC"] = keyvals[2].replace("kfac=", "") == "true"
        df["Damping"] = float(keyvals[3].replace("damping=", ""))
        df["Metric Power"] = float(keyvals[4].replace("metricpow=", ""))

        dfs_list.append(df)

    return pd.concat(dfs_list).reset_index()


experiment_df = get_experiment_df()

In [ ]:
plot_df = experiment_df.copy()
plot_df["Method"] = plot_df.apply(
    lambda row: f"{('K-FAC' if row['K-FAC'] else 'Shampoo')}^{row['Metric Power']} + {row['Damping']}",
    axis=1,
)
plot_df = plot_df[2000 <= plot_df["step"]]
plot_df = plot_df[plot_df["eval/accuracy"] > 0.75]

sns_opts = {
    "hue": "Method",
    "hue_order": sorted(plot_df["Method"].unique(), key=lambda x: (float(x.split("+")[1].strip()), x)),
    "errorbar": "sd",
}

lrs = sorted(plot_df["LR"].unique())
fig, ax = plt.subplots(1, len(lrs), figsize=(2 + 3 * len(lrs), 5))
min_accuracy = plot_df["eval/accuracy"].min()
for i, lr in enumerate(lrs):
    sns.lineplot(
        ax=ax[i],
        data=plot_df[plot_df["LR"] == lr],
        x="step",
        y="eval/accuracy",
        **sns_opts,
    )
    ax[i].grid(True, which="both", linestyle="--", linewidth=0.5)
    ax[i].set_title(f"LR = {lr}")
    ax[i].set_ylabel("Accuracy")
    ax[i].set_xlabel("Steps")
    ax[i].set_ylim(min_accuracy - 0.01)

fig.subplots_adjust(bottom=0.2)
plt.suptitle("Scaled PSI-LoRA on GLUE-MNLI")
fig.tight_layout()

plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

In [ ]:
plot_df = experiment_df.copy()
plot_df["Method"] = plot_df.apply(
    lambda row: f"{('K-FAC' if row['K-FAC'] else 'Shampoo')}^{row['Metric Power']} + {row['Damping']}",
    axis=1,
)
plot_df = plot_df[2000 <= plot_df["step"]]

sns_opts = {
    "hue": "Method",
    "hue_order": sorted(plot_df["Method"].unique(), key=lambda x: (float(x.split("+")[1].strip()), x)),
    "style": "LR",
    "style_order": sorted(plot_df["LR"].unique(), reverse=True),
    "errorbar": "sd",
}

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
min_accuracy = plot_df["eval/accuracy"].min()
sns.lineplot(
    ax=ax[0],
    data=plot_df,
    x="step",
    y="eval/accuracy",
    **sns_opts,
)
ax[0].grid(True, which="both", linestyle="--", linewidth=0.5)
ax[0].set_ylabel("Accuracy")
ax[0].set_xlabel("Training Steps")

sns.lineplot(
    ax=ax[1],
    data=plot_df,
    x="step",
    y="eval/loss",
    **sns_opts,
)
ax[1].grid(True, which="both", linestyle="--", linewidth=0.5)
ax[1].set_ylabel("Eval Loss")
ax[1].set_xlabel("Training Steps")

fig.subplots_adjust(bottom=0.2)
plt.suptitle("Scaled PSI-LoRA on GLUE-MNLI")
fig.tight_layout()

# plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

In [ ]:
plot_df.loc[plot_df["step"] == plot_df["step"].max(), ["Method", "LR", "eval/accuracy", "eval/loss"]].sort_values("eval/accuracy", ascending=False)